In [ ]:
import polars as pl
import httpx
from dotenv import load_dotenv
import os

load_dotenv()

PURPLE_AIR_API_KEY = os.getenv("PURPLE_AIR_API_KEY")

In [ ]:
# get json data from purple air response - note this code is from httpx documentation and just modified to work with this project's usecase

try:
    sensors = httpx.get(
        "https://api.purpleair.com/v1/sensors",
        params={
            "api_key": PURPLE_AIR_API_KEY,
            "fields": "sensor_index,latitude,longitude,position_rating,location_type,last_seen,pm2.5_atm,temperature,humidity",
            "max_age": 0,
            "nwlat": 40.962891,
            "nwlng": -74.264395,
            "selat": 40.470116,
            "selng": -73.675450,
        },
    )
    if sensors.status_code == 200:
        sensors_json = sensors.json()
        print(sensors_json)
    else:
        print(f"Request failed with status code: {sensors.status_code}, response: {sensors.text}")
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to make the request: {e}")

In [ ]:
raw_sensors_df = pl.DataFrame(data = sensors_json["data"], \
                  schema = sensors_json["fields"])

print(raw_sensors_df.height)
raw_sensors_df.head()

### Cleaning data

Since we are looking to create a dashboard that shows air quality in NYC, we need to firstly clean the data and do a spatial join to get sensors only within the 5 boroughs,
and also remove the sensors which are in indoors (location_type = 1) and those with a position rating of 0 (position_rating = 0) since they are likely to be inaccurate. We will also convert the last_seen column to a datetime format.

In [ ]:
import geopandas as gpd

raw_sensors_gdf = gpd.GeoDataFrame(
                    raw_sensors_df.to_pandas(),
                    geometry=gpd.points_from_xy(raw_sensors_df["longitude"], raw_sensors_df["latitude"]),
                    crs="EPSG:4326"
                )

raw_sensors_gdf = raw_sensors_gdf.loc[(raw_sensors_gdf["position_rating"] > 0) & (raw_sensors_gdf["location_type"] != 1)]

raw_sensors_gdf.head()

In [ ]:
# Now that the data is in a geodataframe, we can do a spatial join to remove sensors outside of the 5 boroughs of NYC.
# The borough boundaries data is in the NYC Open Data portal.

response = httpx.get("https://data.cityofnewyork.us/resource/gthc-hcne.geojson")
response.raise_for_status()

geojson = response.json()

boroughs_gdf = gpd.GeoDataFrame.from_features(geojson["features"], crs="EPSG:4326")
boroughs_gdf.head()


In [ ]:
# Get Sensors Data API End Point - to be used in Kestra Flow
# Store a wide-but-reasonable schema in the lake so dbt can build 'typical pattern' aggregates (time-of-day/season/borough).
#
# Column intent (high-level):
# - IDs + geo: keys + spatial join to NYC borough boundaries + clustering in BigQuery
# - Status/filtering: exclude indoor sensors + low-quality location (position_rating) + private sensors if desired
# - Timestamps: last_seen drives event-time; last_modified/date_created help debugging changes/new sensors
# - QA flags + confidence: exclude downgraded channels / suspicious readings when building 'typical' averages
# - PM + env metrics: the actual dashboard measures + covariates for seasonality/time-of-day patterns
# - Device health: troubleshoot sensors with latency/weak WiFi (helps explain weird values or dropouts)
#
# Note on naming: PurpleAir includes dots in some field names (e.g. pm2.5_atm).
# When landing to BigQuery, you’ll typically normalize column names (e.g. replace '.' with '_') in ingestion or dbt.

try:
    sensors = httpx.get(
        "https://api.purpleair.com/v1/sensors",
        params={
            "api_key": PURPLE_AIR_API_KEY,
            "fields": (
                # IDs + geo
                "sensor_index,name,latitude,longitude,altitude,"
                # Status / filtering
                "location_type,private,position_rating,"
                # Timestamps / provenance
                "last_seen,last_modified,date_created,"
                # QA: channel downgrade flags + confidence
                "channel_state,channel_flags,channel_flags_manual,channel_flags_auto,"
                "confidence,confidence_manual,confidence_auto,"
                # Core air quality metrics (mass concentration)
                "pm1.0_atm,pm2.5_atm,pm2.5_atm_a,pm2.5_atm_b,pm2.5_alt,pm10.0_atm,"
                # Environmental context
                "humidity,temperature,pressure,"
                # Device health / troubleshooting
                "rssi,uptime,pa_latency"
            ),
            "max_age": 0,
            "nwlat": 40.962891,
            "nwlng": -74.264395,
            "selat": 40.470116,
            "selng": -73.675450,
        },
        timeout=30.0,
    )
    if sensors.status_code == 200:
        sensors_json = sensors.json()
        print(sensors_json)
    else:
        print(f"Request failed with status code: {sensors.status_code}, response: {sensors.text}")
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to make the request: {e}")